Preparation des données a partir de modele3.csv

In [1]:
import pandas as pd
df = pd.read_csv("modele3v7.csv")
df.columns

Index(['film_id', 'realisateur_id', 'rang', 'title', 'titre_vo', 'realisateur',
       'genre', 'annee', 'pays', 'entrees', 'salles', 'moy_salle',
       'part_marche', 'affiche', 'moyenne_fr_realisateur', 'sortie',
       'distributeur', 'classification', 'acteurs',
       'moyennes_individuelles_acteurs', 'acteur_principal',
       'acteur_secondaire', 'max_moyenne', 'moyennebestactors', 'Pays',
       'cluster_realisateur', 'cluster_acteur_principal',
       'cluster_acteur_secondaire', 'box_office_fr', 'box_office_us',
       'audience', 'budget', 'voix_off', 'demarrage', 'writer', 'languages',
       'sommedesmoyennes', 'duration', 'cluster_acteur',
       'somme_cluster_acteur'],
      dtype='object')

nettoyage

In [18]:
# ------------------------------------------------------------
# 0.  Librairies
# ------------------------------------------------------------
import re
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from typing import List

# ------------------------------------------------------------
# 1.  Nettoyeur / transformeur personnalisé
# ------------------------------------------------------------
class FilmDataCleaner(BaseEstimator, TransformerMixin):
    """
    - Filtre les films sortis à partir de 2000
    - Nettoie les chaînes numériques mal formatées
    - Supprime les colonnes qui fuient la cible
    - Construit `age_rating` (audience > classification)
    """
    def __init__(self,
                 target_col: str = "demarrage",
                 leak_cols: List[str] = None):
        self.target_col = target_col
        self.leak_cols = leak_cols or [
            "moy_salle", "part_marche",
            "entrees", "box_office_fr", "box_office_us"
        ]

    # -----------------------------------------------------------------
    #  (aucun apprentissage -> fit retourne simplement self)
    # -----------------------------------------------------------------
    def fit(self, X: pd.DataFrame, y=None):
        return self

    # -----------------------------------------------------------------
    #  Fonction de nettoyage interne
    # -----------------------------------------------------------------
    @staticmethod
    def _clean_numeric_series(s: pd.Series) -> pd.Series:
        """Convertit une série objet en numérique si tout (hors NaN) est nombre."""
        cleaned = (s.astype(str)
                     .str.replace(r"\s+", "", regex=True)  # suppr. espaces & nbsp
                     .str.replace("%", "")
                     .str.replace(",", ".")
                     .replace("", np.nan))
        if cleaned.dropna().str.fullmatch(r"-?\d+(\.\d+)?").all():
            return pd.to_numeric(cleaned, errors="coerce")
        return s

    # -----------------------------------------------------------------
    #  Transformation principale
    # -----------------------------------------------------------------
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        df = X.copy()

        # --- 1. Filtre année -----------------------------------------
        if "annee" in df.columns:
            df = df[df["annee"] >= 2000]

        # --- 2. Nettoyage chaînes numériques -------------------------
        for col in df.select_dtypes(include="object").columns:
            df[col] = self._clean_numeric_series(df[col])

        # --- 3. Suppression target‑leak ------------------------------
        df.drop(columns=[c for c in self.leak_cols if c in df.columns],
                inplace=True, errors="ignore")

        # --- 4. Fusion audience / classification --------------------
        if "audience" in df.columns or "classification" in df.columns:
            df["age_rating"] = df.get("audience").fillna(df.get("classification"))
            df.drop(columns=["audience", "classification"],
                    inplace=True, errors="ignore")

        # --- 5. Retour (la cible n'est PAS retirée ici) -------------
        # On laisse encore la colonne target pour que l'appelant la sépare.
        return df.reset_index(drop=True)

# ------------------------------------------------------------
# 2.  Construction du pipeline de pré‑traitement
# ------------------------------------------------------------
# À ce stade, seul le nettoyage est encapsulé.
# Les étapes d'encodage et de normalisation viendront ensuite.
preprocess_pipeline = Pipeline(steps=[
    ("cleaner", FilmDataCleaner())
])

# ------------------------------------------------------------
# 3.  Exemple d’utilisation
# ------------------------------------------------------------
if __name__ == "__main__":
    CSV_PATH = "modele3v7.csv"          # adapter le chemin
    raw_df = pd.read_csv(CSV_PATH)

    # a) pipeline fit_transform (rien n'est appris, mais on garde la syntaxe)
    clean_df = preprocess_pipeline.fit_transform(raw_df)

    # b) Séparation features / target pour l'étape 2
    TARGET_COL = "demarrage"
    X = clean_df.drop(columns=[TARGET_COL])
    y = clean_df[TARGET_COL]

    print("Films conservés :", len(clean_df))
    print("Variables (features) :", X.shape[1])
    print("Aperçu :", X.head(3).T)


Films conservés : 7395
Variables (features) : 33
Aperçu :                                                                                 0  \
film_id                                                                     13345   
realisateur_id                                                              575.0   
rang                                                                            3   
title                                            Star Wars: Le Réveil de la Force   
titre_vo                                             Star Wars: The Force Awakens   
realisateur                                                           J.J. Abrams   
genre                                                                     Fantasy   
annee                                                                        2015   
pays                                                                          f23   
salles                                                                       1093   
affiche

In [20]:
# =====================================================================
# 0.  Librairies
# =====================================================================
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn import __version__ as skl_version
from packaging import version
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import KFold, train_test_split
from xgboost import XGBRegressor
import optuna

# ---------- RMSE (compatibilité anciennes versions) -------------------
try:                                   # scikit‑learn ≥ 1.5
    from sklearn.metrics import root_mean_squared_error as rmse
except ImportError:                    # scikit‑learn < 1.5
    from sklearn.metrics import mean_squared_error
    rmse = lambda y, yhat: np.sqrt(mean_squared_error(y, yhat))

# =====================================================================
# 1.  FilmDataCleaner
# =====================================================================
class FilmDataCleaner(BaseEstimator, TransformerMixin):
    """Nettoyage + filtrage (année ≥ 2000) + suppression target‑leak."""

    def __init__(self,
                 target_col: str = "demarrage",
                 leak_cols=None):
        self.target_col = target_col
        self.leak_cols = leak_cols or [
            "moy_salle", "part_marche",
            "entrees", "box_office_fr", "box_office_us"
        ]

    def fit(self, X, y=None):  # rien à apprendre
        return self

    @staticmethod
    def _clean_num(s: pd.Series) -> pd.Series:
        cleaned = (s.astype(str)
                     .str.replace(r"\s+", "", regex=True)
                     .str.replace("%", "")
                     .str.replace(",", ".")
                     .replace("", np.nan))
        if cleaned.dropna().str.fullmatch(r"-?\d+(\.\d+)?").all():
            return pd.to_numeric(cleaned, errors="coerce")
        return s

    def transform(self, X):
        df = X.copy()

        # 1) films depuis 2000
        if "annee" in df:
            df = df[df["annee"] >= 2000]

        # 2) nettoyage numérique
        for col in df.select_dtypes("object").columns:
            df[col] = self._clean_num(df[col])

        # 3) suppression colonnes qui fuient la cible
        df.drop(columns=[c for c in self.leak_cols if c in df.columns],
                inplace=True, errors="ignore")

        # 4) fusion audience / classification
        if "audience" in df or "classification" in df:
            df["age_rating"] = df.get("audience").fillna(df.get("classification"))
            df.drop(columns=["audience", "classification"],
                    inplace=True, errors="ignore")

        return df.reset_index(drop=True)

# =====================================================================
# 2.  Chargement + nettoyage
# =====================================================================
raw_df   = pd.read_csv(Path("modele3v7.csv"))
cleaner  = FilmDataCleaner()
clean_df = cleaner.fit_transform(raw_df)

TARGET   = "demarrage"
X_all    = clean_df.drop(columns=[TARGET])
y_all    = clean_df[TARGET]

# =====================================================================
# 3.  Poids blockbusters (top 20 %  → poids 4)
# =====================================================================
p80            = y_all.quantile(0.80)
sample_weights = np.where(y_all >= p80, 4.0, 1.0)

# =====================================================================
# 4.  Encodage : One‑Hot sparse + StandardScaler
# =====================================================================
num_cols = selector(dtype_include=["int64", "float64"])(X_all)
cat_cols = selector(dtype_include=["object", "category", "bool"])(X_all)

if version.parse(skl_version) >= version.parse("1.4"):
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
else:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", ohe,             cat_cols)
])

# =====================================================================
# 5.  Fonction objectif Optuna (boucle k‑fold + sample_weight)
# =====================================================================
def objective(trial):

    params = dict(
        n_estimators     = trial.suggest_int("n_estimators", 200, 800),
        learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_depth        = trial.suggest_int("max_depth", 4, 10),
        min_child_weight = trial.suggest_float("min_child_weight", 1, 10),
        subsample        = trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0),
        gamma            = trial.suggest_float("gamma", 0, 5),
        reg_alpha        = trial.suggest_float("reg_alpha", 0, 5),
        reg_lambda       = trial.suggest_float("reg_lambda", 0, 5),
        objective        = "reg:squarederror",
        eval_metric      = "rmse",          # XGBoost ≥ 2.0
        n_jobs           = -1,
        random_state     = 42,
    )

    pipe = Pipeline([
        ("clean", cleaner),
        ("prep",  preprocessor),
        ("xgb",   XGBRegressor(**params))
    ])

    cv   = KFold(n_splits=5, shuffle=True, random_state=42)
    rms  = []

    for tr_idx, vl_idx in cv.split(X_all):
        X_tr, X_vl = X_all.iloc[tr_idx], X_all.iloc[vl_idx]
        y_tr, y_vl = y_all.iloc[tr_idx], y_all.iloc[vl_idx]
        w_tr       = sample_weights[tr_idx]

        pipe.fit(X_tr, y_tr, xgb__sample_weight=w_tr)
        rms.append(rmse(y_vl, pipe.predict(X_vl)))

    return np.mean(rms)

# =====================================================================
# 6.  Optimisation Optuna
# =====================================================================
study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Meilleurs hyper‑paramètres :", study.best_params)
print("RMSE CV moyenne           :", study.best_value)

# =====================================================================
# 7.  Entraînement final + hold‑out 80/20
# =====================================================================
best_params = study.best_params | {
    "objective":  "reg:squarederror",
    "eval_metric": "rmse",
    "n_jobs":     -1,
    "random_state": 42
}

final_pipe = Pipeline([
    ("clean", cleaner),
    ("prep",  preprocessor),
    ("xgb",   XGBRegressor(**best_params))
])

X_tr, X_te, y_tr, y_te, w_tr, _ = train_test_split(
    X_all, y_all, sample_weights, test_size=0.2, random_state=42
)

final_pipe.fit(X_tr, y_tr, xgb__sample_weight=w_tr)
print(f"RMSE hold‑out finale : {rmse(y_te, final_pipe.predict(X_te)):,.0f}")

# =====================================================================
# 8.  (Option) sauvegarde du pipeline
# =====================================================================
import joblib
joblib.dump(final_pipe, "xgb_demarrage_optuna.joblib")


[I 2025-04-18 00:25:56,019] A new study created in memory with name: no-name-90ec87ed-b45f-473b-85e8-78e178616964


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-04-18 00:27:02,455] Trial 0 finished with value: 182198.469006863 and parameters: {'n_estimators': 425, 'learning_rate': 0.2536999076681772, 'max_depth': 9, 'min_child_weight': 6.387926357773329, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.5779972601681014, 'gamma': 0.2904180608409973, 'reg_alpha': 4.330880728874676, 'reg_lambda': 3.005575058716044}. Best is trial 0 with value: 182198.469006863.
[I 2025-04-18 00:30:11,871] Trial 1 finished with value: 165395.87729361438 and parameters: {'n_estimators': 625, 'learning_rate': 0.010725209743171997, 'max_depth': 10, 'min_child_weight': 8.491983767203795, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.5909124836035503, 'gamma': 0.9170225492671691, 'reg_alpha': 1.5212112147976886, 'reg_lambda': 2.6237821581611893}. Best is trial 1 with value: 165395.87729361438.
[I 2025-04-18 00:31:24,953] Trial 2 finished with value: 163290.29631517213 and parameters: {'n_estimators': 459, 'learning_rate': 0.02692655251486473, 'ma

['xgb_demarrage_optuna.joblib']

In [21]:
from joblib import load
cine = load("xgb_demarrage_optuna.joblib")
print(cine)

Pipeline(steps=[('clean',
                 FilmDataCleaner(leak_cols=['moy_salle', 'part_marche',
                                            'entrees', 'box_office_fr',
                                            'box_office_us'])),
                ('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['film_id', 'realisateur_id',
                                                   'annee', 'salles',
                                                   'moyenne_fr_realisateur',
                                                   'max_moyenne',
                                                   'moyennebestactors',
                                                   'cluster_realisateur',
                                                   'cluster_acteur_principal',
                                                   'cluster_a...
                              gamma=0.6155467087603368, grow_policy=None,
         

In [23]:
import json
import pandas as pd

# On suppose que 'study' contient déjà la recherche Optuna

best = study.best_trial

print(f"→ Trial n°{best.number}  |  RMSE CV = {best.value:.0f}")
print("→ Hyper‑paramètres optimaux :")
for k, v in best.params.items():
    print(f"   • {k} = {v}")



→ Trial n°30  |  RMSE CV = 162488
→ Hyper‑paramètres optimaux :
   • n_estimators = 600
   • learning_rate = 0.03953161674926092
   • max_depth = 5
   • min_child_weight = 3.426785665254986
   • subsample = 0.6373258511602691
   • colsample_bytree = 0.7417902398612948
   • gamma = 0.6155467087603368
   • reg_alpha = 2.619264958694323
   • reg_lambda = 1.1811761907506746
